# **Séance 7 : Du tableur à Python : CSV**

Jusqu'ici, notre corpus de bestiaires était une liste de dictionnaires. Dans ce notebook, nous le retrouvons sous la forme d'un **tableau** : le fichier `corpus.csv`, qui décrit 15 manuscrits de bestiaires et de textes apparentés (*Physiologus*, *Aviarium*).

Chaque ligne décrit un manuscrit, en 9 colonnes : `sigle`, `version`, `ville`, `bibliotheque`, `cote`, `date`, `folios`, `commentaire` et `manifeste` (l'adresse de son manifeste IIIF, vu en séance 6).

Avant de commencer, placez `corpus.csv` dans un dossier `donnees`, à côté de ce notebook.

---

## **0. Préparation**

In [ ]:
import pandas as pd
from pathlib import Path

print(pd.__version__)

fichier = Path("donnees") / "corpus.csv"
print(fichier.exists())

Si la version commence par `2`, certains affichages diffèrent légèrement de ceux des slides : les colonnes de texte y sont de type `object` au lieu de `str`.

---

## **1. Le CSV : un tableur en texte brut**

Avant d'utiliser pandas, ouvrons le fichier comme en séance 5, avec `open()`. Un CSV n'est que du texte : `splitlines()` le découpe en une liste de lignes.

In [ ]:
with open(fichier, encoding="utf-8") as f:
    lignes = f.read().splitlines()

print(len(lignes))
for ligne in lignes[:4]:
    print(ligne)

On compte 16 lignes : l'en-tête, puis une notice par manuscrit. Les cellules sont séparées par des points-virgules, et deux points-virgules qui se suivent (`;;`) signalent une cellule vide.

---

## **2. Les pièges du CSV**

Si on ne précise rien, `read_csv()` suppose que le séparateur est une **virgule**. Essayons. *La cellule suivante doit provoquer une erreur.*

In [ ]:
df = pd.read_csv(fichier)

Lisez la dernière ligne du message : pandas s'arrête à la **ligne 14**, celle de Merton College, dont le commentaire contient une virgule. Avec le bon séparateur, la lecture fonctionne :

In [ ]:
df = pd.read_csv(fichier, sep=";")
print(df.shape)

Notre fichier ne contient pas de nombres décimaux. S'il en contenait, écrits à la française (`30,5`), il faudrait ajouter `decimal=","`.

---

## **3. Une question, deux réponses**

Pour comparer avec ce que vous savez déjà faire, voici le corpus sous sa forme habituelle, une liste de dictionnaires :

In [ ]:
corpus = df.to_dict("records")   # conversion fournie pour l'exercice
print(corpus[0])

#### **Exercice 1 : Compter les manuscrits par sigle**

1. Avec une boucle `for` et un dictionnaire `compte`, comptez le nombre de manuscrits pour chaque `sigle` de `corpus`.
2. Obtenez le même résultat avec pandas, en une ligne.

*Solution :*

In [ ]:
# Votre code ici

---

## **4. Lire et explorer un tableau**

##### **Afficher un extrait**

In [ ]:
df.head()

In [ ]:
df.tail(3)

##### **Examiner la structure**

In [ ]:
print(df.shape)
print(df.columns)

##### **Faire le bilan des colonnes et résumer les valeurs**

In [ ]:
df.info()

In [ ]:
df.describe()

#### **Exercice 2 : Faire le portrait du corpus**

En vous aidant des cellules précédentes, répondez en commentaire :

1. Combien de manuscrits et de colonnes compte le tableau ?
2. Quelles colonnes contiennent des valeurs manquantes, et combien ?
3. De quel type est la colonne `date` ? Pourquoi, d'après vous ?
4. Quel est le sigle le plus fréquent ? Quelle ligne de `describe()` vous l'indique ?

*Solution :*

In [ ]:
# Votre code ici

---

## **5. Sélectionner et filtrer**

##### **Sélectionner des colonnes**

In [ ]:
df["cote"]

In [ ]:
df[["cote", "ville"]]

##### **Sélectionner des lignes : `loc` et `iloc`**

In [ ]:
print(df.loc[8, "cote"])      # par étiquette
print(df.iloc[0]["cote"])      # par position

##### **Filtrer avec une condition**

In [ ]:
df[df["ville"] == "London"]

Les lignes filtrées gardent leur **étiquette** : étiquettes et positions ne coïncident plus.

In [ ]:
londres = df[df["ville"] == "London"]

print(londres.iloc[0]["cote"])    # la première ligne du résultat
print(londres.loc[2, "cote"])     # la ligne d'étiquette 2 : la même

En revanche, aucune ligne de `londres` ne porte l'étiquette `0` (c'est Berne, à Bern). *La cellule suivante doit provoquer une erreur.*

In [ ]:
londres.loc[0]

#### **Exercice 3 : Les manuscrits de la British Library**

Affichez la cote et la date des manuscrits conservés à la British Library (`"BL"` dans la colonne `bibliotheque`).

*Solution :*

In [ ]:
# Votre code ici

#### **Exercice 4 : La Seconde Famille à Londres et à Oxford**

Affichez la cote et la ville des manuscrits de la Seconde Famille (sigle `"2F"`) conservés à Londres **ou** à Oxford.

*Indice :* il faut combiner `&` et `|`. Attention aux parenthèses.

*Solution :*

In [ ]:
# Votre code ici

---

## **6. Nettoyer les données**

##### **Repérer les valeurs manquantes**

In [ ]:
df.isna().sum()

In [ ]:
df.loc[df["manifeste"].isna(), ["cote", "ville"]]

##### **Supprimer les lignes incomplètes : attention !**

In [ ]:
print(df.dropna().shape)
print(df.dropna(subset=["manifeste"]).shape)

Sans précision, `dropna()` ne garde que les 3 manuscrits dont **toutes** les cellules sont remplies. Avec `subset`, on ne vérifie que la colonne choisie.

#### **Exercice 5 : Remplacer les versions manquantes**

1. Remplacez les valeurs manquantes de la colonne `version` par `"inconnue"`, en gardant le résultat dans `df`.
2. Vérifiez qu'il ne reste plus de valeur manquante dans cette colonne.

*Solution :*

In [ ]:
# Votre code ici

##### **Convertir un type**

La colonne `date` contient du texte. *La cellule suivante doit provoquer une erreur.*

In [ ]:
df["date"].astype(int)

In [ ]:
df["date"].str[:2]

#### **Exercice 6 : La colonne `siecle`**

1. Créez une colonne `siecle` qui contient le siècle de départ, sous forme de nombre.
2. Affichez la cote, la date et le siècle des manuscrits du XIIIᵉ siècle et après.
3. Comptez les manuscrits par siècle.
4. Affichez la cote et le siècle de tous les manuscrits, du plus ancien au plus récent.

*Solution :*

In [ ]:
# Votre code ici

---

## **7. Écrire un fichier**

#### **Exercice 7 : Enregistrer les manuscrits du XIIᵉ siècle**

1. Enregistrez la cote, la ville et le siècle des manuscrits du XIIᵉ siècle dans `donnees/xiie_siecle.csv`, avec un point-virgule comme séparateur et sans l'index.
2. Relisez le fichier avec `read_csv()` pour vérifier son contenu.

*Solution :*

In [ ]:
# Votre code ici

---

## **Exercice de synthèse**

#### **Exercice 8 : La liste des manifestes**

En séance 6, nous avons interrogé les manifestes IIIF un par un. Préparons la liste complète.

En repartant de `corpus.csv` :

1. Relisez le fichier dans `df`.
2. Ajoutez la colonne `siecle`.
3. Ne gardez que les manuscrits qui ont un manifeste.
4. Triez-les du plus ancien au plus récent.
5. Ne gardez que les colonnes `cote`, `ville`, `bibliotheque`, `siecle` et `manifeste`.
6. Enregistrez le résultat dans `donnees/manifestes.csv`, puis relisez-le pour vérifier.

*Solution :*

In [ ]:
# Votre code ici

---

### **Conclusion**

Dans ce notebook, nous avons manipulé notre corpus sous la forme d'un tableau :

- **lire** un CSV avec `read_csv()`, en précisant le séparateur ;
- **explorer** un tableau avec `head()`, `info()`, `describe()`, `shape` et `columns` ;
- **sélectionner** des colonnes et des lignes avec `[]`, `loc` et `iloc`, et **filtrer** avec des conditions ;
- **nettoyer** les valeurs manquantes, **créer** une colonne, **trier** et **compter** ;
- **écrire** un nouveau CSV avec `to_csv()`, lisible dans un tableur comme dans Python.